# build-raw-tissdiss-v2
Per-sample QC → concat → normalize → UMAP. All input files from `../../input/`.

In [1]:
import glob, re, os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scanpy.external as sce
from scipy.stats import median_abs_deviation
from concurrent.futures import ThreadPoolExecutor

sc.settings.verbosity = 1

In [2]:
def is_outlier(adata, metric, nmads=5):
  M = adata.obs[metric]
  return (M < np.median(M) - nmads * median_abs_deviation(M)) | (M > np.median(M) + nmads * median_abs_deviation(M))


def process_sample(path):
  fname       = os.path.basename(path)
  source_file = fname.replace('_sample_filtered_feature_bc_matrix.h5', '')

  adata = sc.read_10x_h5(path)
  adata.var_names_make_unique()
  adata.obs['source_file']      = source_file
  adata.obs['original_barcode'] = adata.obs.index.astype(str)

  adata.var['mt'] = adata.var_names.str.startswith('MT-')
  sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=[20], log1p=True, inplace=True)

  adata.obs['outlier'] = (
      is_outlier(adata, 'log1p_total_counts')
      | is_outlier(adata, 'log1p_n_genes_by_counts')
      | is_outlier(adata, 'pct_counts_in_top_20_genes')
  )

  record = {
      'source_file':        source_file,
      'n_cells_raw':        adata.n_obs,
      'n_outliers_flagged': int(adata.obs['outlier'].sum()),
      'median_n_genes':     int(np.median(adata.obs['n_genes_by_counts'])),
      'median_n_counts':    int(np.median(adata.obs['total_counts'])),
      'median_pct_mt':      round(float(np.median(adata.obs['pct_counts_mt'])), 2),
  }

  return adata, record

In [3]:
metadata = pd.read_csv('../files/updated_metadata.csv')
pcd_store_files = list(metadata[metadata['flag'].isna()]['file_path'])

In [4]:
with ThreadPoolExecutor(max_workers=16) as ex:
    results = list(ex.map(process_sample, pcd_store_files))

adata_list = [r[0] for r in results]
qc_records = [r[1] for r in results]

for rec in qc_records:
    print(f"{rec['source_file']}: {rec['n_cells_raw']} cells "
          f"({rec['n_outliers_flagged']} outliers flagged ")

qc_df = pd.DataFrame(qc_records)
print(f"\nTotal: {qc_df['n_cells_raw'].sum()} raw | {qc_df['n_outliers_flagged'].sum()} Outliers")

EXP01109_Multiplex1_5_TIS05684_3: 13863 cells (50 outliers flagged 
Exp998_8_chronic_myleogenous_leukemia_TIS05398-001-008: 799 cells (16 outliers flagged 
EXP01387_3_TIS09206-001-001: 114 cells (4 outliers flagged 
EXP01109_Multiplex1_1_TIS06392_1: 9858 cells (12 outliers flagged 
EXP01109_TIS05681-001-004_c2: 12229 cells (110 outliers flagged 
EXP01387_7_TIS09210-001-001: 3423 cells (127 outliers flagged 
EXP01109_Multiplex2_5_TIS05684_1: 8657 cells (46 outliers flagged 
EXP01109_TIS05681-001-004_c1: 13026 cells (148 outliers flagged 
EXP01109_Multiplex1_4_TIS05683_2: 14976 cells (387 outliers flagged 
EXP01109_Multiplex2_5_TIS05684_3: 12145 cells (62 outliers flagged 
EXP01109_TIS05684-001-004_c2: 9289 cells (69 outliers flagged 
EXP01109_Multiplex1_3_TIS05681_3: 12856 cells (88 outliers flagged 
EXP00998_1_MM_TIS05392-001-008: 1213 cells (63 outliers flagged 
EXP01109_Multiplex1_1_TIS06392_3: 13886 cells (17 outliers flagged 
Exp998_6_early_myoproliferative_TIS05395-001-009: 473 ce

In [5]:
qc_df = qc_df.merge(metadata, on='source_file', how='left', suffixes=('', '_meta'))

meta_cols = [c for c in metadata.columns if c not in ('source_file', 'TIS_ID', 'project')]
for adata, (_, row) in zip(adata_list, qc_df.iterrows()):
  for col in meta_cols:
      adata.obs[col] = row[col]

In [6]:
qc_df.to_csv('../files/qc-summary.csv', index=False)

In [7]:
adata_merged = ad.concat(adata_list, join='outer')
adata_merged.var_names_make_unique()
adata_merged.obs_names_make_unique()
adata_merged.raw = adata_merged
adata_merged

AnnData object with n_obs × n_vars = 556792 × 18593
    obs: 'source_file', 'TIS_ID', 'project', 'original_barcode', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'outlier', 'file_path', 'TIS_ID_base', 'in_cohort_metadata', 'flag', 'subject_id', 'cohort_label', 'tissue_type', 'sample_site', 'disease_state', 'age_at_collection', 'sex', 'race_ethnicity', 'treatment', 'pct_tumor', 'collection_year', 'biopsy_or_resection'

In [8]:
# build label map from old object — keyed on original_barcode (pre-uniquify suffix)
old_labels = pd.read_parquet("../files/old_labels.parquet")['manual_map_leiden'].to_dict()
adata_merged.obs['old_labels'] = adata_merged.obs['original_barcode'].map(old_labels)

matched   = adata_merged.obs['old_labels'].notna().sum()
unmatched = adata_merged.obs['old_labels'].isna().sum()
print(f"Matched: {matched} | Unmatched (new cells): {unmatched}")

Matched: 336570 | Unmatched (new cells): 220222


In [9]:
adata_merged.write('../files/raw-all-tissdiss.h5ad')

In [10]:
adata_merged

AnnData object with n_obs × n_vars = 556792 × 18593
    obs: 'source_file', 'TIS_ID', 'project', 'original_barcode', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'outlier', 'file_path', 'TIS_ID_base', 'in_cohort_metadata', 'flag', 'subject_id', 'cohort_label', 'tissue_type', 'sample_site', 'disease_state', 'age_at_collection', 'sex', 'race_ethnicity', 'treatment', 'pct_tumor', 'collection_year', 'biopsy_or_resection', 'old_labels'